# 04 — Validate ohsome API responses on the 10-cell test sample

This notebook performs real requests against the current ohsome API for the deterministic 10-cell sample prepared in Notebook 03.

It is deliberately a validation notebook. It:
1. records `/metadata` and the OSHDB temporal extent;
2. queries road length and current road-element count at the latest OSHDB snapshot;
3. queries unique users and contributions over the full OSHDB history;
4. persists every successful response and every HTTP error body;
5. validates endpoint-specific response schemas before parsing;
6. produces one tidy 10-cell validation table keyed by `grid_1km_id`.

No request is made for the full set of 895 cells in this notebook.


In [ ]:
from pathlib import Path
import json
import time
from datetime import datetime, timezone
import pandas as pd
import geopandas as gpd
import requests

API_BASE = "https://api.ohsome.org/v1"
ROAD_FILTER = "highway=* and geometry:line"
REQUEST_TIMEOUT = 180
TRANSIENT_STATUS = {429, 500, 502, 503, 504}

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, start.parent, *start.parents]:
        if (candidate / "notebooks").exists() and (candidate / "Petropolis").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root.")

PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_DIR = PROJECT_ROOT / "data" / "intermediate" / "ohsome" / "Petropolis"
TEST_BOUNDARIES_FILE = INPUT_DIR / "petropolis_grid_1km_ohsome_test_boundaries.geojson"
TEST_INDEX_FILE = INPUT_DIR / "petropolis_grid_1km_ohsome_test_cells.parquet"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ohsome" / "Petropolis" / "test_10_cells"
RESULTS_DIR = PROJECT_ROOT / "data" / "processed" / "Petropolis"
RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_FILE = RESULTS_DIR / "petropolis_ohsome_test_10_cells.parquet"

for p in (TEST_BOUNDARIES_FILE, TEST_INDEX_FILE):
    if not p.exists():
        raise FileNotFoundError(f"Required Notebook 03 output not found: {p}")

session = requests.Session()
session.headers.update({"User-Agent":"sotm-br-2026-research/0.2 (academic OSM analysis)"})


## 1. Read `/metadata` and define temporal parameters


In [ ]:
r = session.get(f"{API_BASE}/metadata", timeout=REQUEST_TIMEOUT)
r.raise_for_status()
metadata = r.json()
(RAW_DIR / "ohsome_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

api_version = metadata.get("apiVersion")
temporal = metadata.get("extractRegion", {}).get("temporalExtent", {})
oshdb_start = temporal.get("fromTimestamp")
oshdb_end = temporal.get("toTimestamp")
if not oshdb_start or not oshdb_end:
    raise ValueError("Could not read extractRegion.temporalExtent from /metadata.")

users_time = f"{oshdb_start}/{oshdb_end}"
contributions_time = f"{oshdb_start},{oshdb_end}"

print("API version:", api_version)
print("OSHDB start:", oshdb_start)
print("OSHDB end:", oshdb_end)
print("Users interval:", users_time)
print("Contributions interval:", contributions_time)


## 2. Load and validate the 10 identified boundaries


In [ ]:
test_cells = gpd.read_parquet(TEST_INDEX_FILE)
feature_collection = json.loads(TEST_BOUNDARIES_FILE.read_text(encoding="utf-8"))
features = feature_collection.get("features", [])
ids = [str(f.get("properties", {}).get("id")) for f in features]

checks = {
    "ten_cells": len(test_cells) == 10,
    "ten_features": len(features) == 10,
    "ids_unique": len(ids) == len(set(ids)),
    "ids_match": set(ids) == set(test_cells["grid_1km_id"].astype(str)),
}
print(checks)
if not all(checks.values()):
    raise ValueError(f"Boundary validation failed: {checks}")

bpolys = json.dumps(feature_collection, ensure_ascii=False, separators=(",", ":"))


## 3. HTTP helper with raw success/error persistence


In [ ]:
def ohsome_post(endpoint, data, stem, max_attempts=4):
    url = f"{API_BASE}{endpoint}"
    last = None
    for attempt in range(1, max_attempts + 1):
        response = session.post(url, data=data, timeout=REQUEST_TIMEOUT)
        content_type = response.headers.get("content-type", "")
        try:
            body = response.json()
        except Exception:
            body = {"raw_text": response.text}

        envelope = {
            "request": {
                "endpoint": endpoint,
                "data_without_bpolys": {k:v for k,v in data.items() if k != "bpolys"},
                "bpolys_feature_count": len(features),
            },
            "response": {
                "status_code": response.status_code,
                "content_type": content_type,
                "body": body,
            },
            "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        suffix = "json" if response.ok else "error.json"
        (RAW_DIR / f"{stem}.{suffix}").write_text(
            json.dumps(envelope, indent=2, ensure_ascii=False), encoding="utf-8"
        )

        if response.ok:
            return body

        last = RuntimeError(
            f"{endpoint} returned HTTP {response.status_code}: "
            f"{json.dumps(body, ensure_ascii=False)[:1200]}"
        )
        if response.status_code not in TRANSIENT_STATUS or attempt == max_attempts:
            raise last
        time.sleep(min(2 ** attempt, 20))
    raise last


## 4. Execute the four endpoint tests


In [ ]:
common = {"bpolys": bpolys, "filter": ROAD_FILTER, "format": "json"}

payloads = {}
payloads["road_length_m"] = ohsome_post(
    "/elements/length/groupBy/boundary",
    common,
    "elements_length_groupby_boundary",
)
payloads["road_element_count"] = ohsome_post(
    "/elements/count/groupBy/boundary",
    common,
    "elements_count_groupby_boundary",
)
payloads["contributors_count"] = ohsome_post(
    "/users/count/groupBy/boundary",
    {**common, "time": users_time},
    "users_count_groupby_boundary",
)
payloads["contributions_count"] = ohsome_post(
    "/contributions/count/groupBy/boundary",
    {**common, "time": contributions_time},
    "contributions_count_groupby_boundary",
)
print("All four HTTP requests returned success.")


## 5. Inspect and validate the real response schemas


In [ ]:
def endpoint_schema(payload):
    groups = payload.get("groupByResult")
    if not isinstance(groups, list):
        return {"groupByResult_list": False}
    sample = None
    for g in groups:
        result = g.get("result")
        if isinstance(result, list) and result:
            sample = result[0]
            break
    return {
        "groupByResult_list": True,
        "groups": len(groups),
        "sample_result_keys": sorted(sample.keys()) if isinstance(sample, dict) else [],
        "apiVersion": payload.get("apiVersion"),
    }

schema_report = {name: endpoint_schema(p) for name,p in payloads.items()}
display(pd.DataFrame(schema_report).T)

expected_ids = set(test_cells["grid_1km_id"].astype(str))
for name, payload in payloads.items():
    groups = payload.get("groupByResult")
    if not isinstance(groups, list):
        raise ValueError(f"{name}: missing groupByResult list.")
    returned_ids = {str(g.get("groupByObject")) for g in groups}
    if returned_ids != expected_ids:
        raise ValueError(
            f"{name}: boundary IDs differ from test sample. "
            f"missing={sorted(expected_ids-returned_ids)}, extra={sorted(returned_ids-expected_ids)}"
        )


## 6. Endpoint-specific parser


In [ ]:
def parse_grouped(payload, metric, temporal_kind):
    rows = []
    for group in payload["groupByResult"]:
        gid = str(group.get("groupByObject"))
        result = group.get("result")
        if not isinstance(result, list) or len(result) != 1:
            raise ValueError(f"{metric}/{gid}: expected exactly one result interval, got {result}")
        item = result[0]
        if "value" not in item:
            raise ValueError(f"{metric}/{gid}: missing value.")
        row = {"grid_1km_id": gid, metric: float(item["value"])}
        if temporal_kind == "snapshot":
            if "timestamp" not in item:
                raise ValueError(f"{metric}/{gid}: expected timestamp.")
            row[f"{metric}_timestamp"] = item["timestamp"]
        elif temporal_kind == "interval":
            if "fromTimestamp" in item and "toTimestamp" in item:
                row[f"{metric}_from"] = item["fromTimestamp"]
                row[f"{metric}_to"] = item["toTimestamp"]
            elif "timestamp" in item:
                row[f"{metric}_timestamp"] = item["timestamp"]
            else:
                raise ValueError(f"{metric}/{gid}: no recognized temporal fields: {item.keys()}")
        rows.append(row)
    return pd.DataFrame(rows)

length_df = parse_grouped(payloads["road_length_m"], "road_length_m", "snapshot")
elements_df = parse_grouped(payloads["road_element_count"], "road_element_count", "snapshot")
users_df = parse_grouped(payloads["contributors_count"], "contributors_count", "interval")
contrib_df = parse_grouped(payloads["contributions_count"], "contributions_count", "interval")


## 7. Consolidate the 10-cell test result and quality-control it


In [ ]:
base_cols = [
    "grid_1km_id", "cell_area_m2", "municipality_fraction",
    "spatial_relation", "source_method", test_cells.geometry.name
]
available = [c for c in base_cols if c in test_cells.columns]
summary = test_cells[available].copy()
summary["grid_1km_id"] = summary["grid_1km_id"].astype(str)

for df in (length_df, elements_df, users_df, contrib_df):
    summary = summary.merge(df, on="grid_1km_id", how="left", validate="one_to_one")

summary["road_length_km"] = summary["road_length_m"] / 1000.0
if "cell_area_m2" in summary.columns:
    summary["road_density_km_per_km2"] = (
        summary["road_length_km"] / (summary["cell_area_m2"] / 1_000_000.0)
    )
summary["ohsome_api_version"] = api_version
summary["oshdb_from"] = oshdb_start
summary["oshdb_to"] = oshdb_end
summary["road_filter"] = ROAD_FILTER
summary["query_run_utc"] = datetime.now(timezone.utc).isoformat()

metric_cols = ["road_length_m","road_element_count","contributors_count","contributions_count"]
qc = {
    "rows_10": len(summary) == 10,
    "grid_id_unique": summary["grid_1km_id"].is_unique,
    "all_metric_values_present": summary[metric_cols].notna().all().all(),
    "all_metric_values_nonnegative": (summary[metric_cols] >= 0).all().all(),
    "all_api_versions_match_metadata": all(
        p.get("apiVersion") == api_version for p in payloads.values()
    ),
}
print(qc)
if not all(qc.values()):
    raise ValueError(f"Final validation failed: {qc}")

summary.to_parquet(SUMMARY_FILE, index=False)
display(summary.drop(columns=[summary.geometry.name], errors="ignore").sort_values("grid_1km_id"))
print("PASS — all four endpoints validated for the 10-cell test sample.")
print("Saved:", SUMMARY_FILE)


## 8. Scale-up gate

The full 895-cell production query may proceed only if this notebook completes without exception.

Validated response contracts are endpoint-specific:
- element length/count: `groupByResult[].result[].timestamp + value`;
- user count: grouped interval result with an accepted temporal representation;
- contribution count: grouped interval result, normally `fromTimestamp + toTimestamp + value`.

Raw success and error envelopes are retained so that a future API patch can be diagnosed without losing the server response.
